# Temporal epistemic read-out: G, F and K over spacetime

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sulcantonin/torchmodal/blob/main/examples/notebooks/02_temporal_epistemic.ipynb)

Worlds do not have to be hypothetical states — they can be **(agent, time)** pairs.
This notebook builds a spacetime Kripke frame for 2 agents over 3 timesteps and reads
off the temporal operators **G** (globally), **F** (finally) and the epistemic **K**
(knows), then shows what the composite `K(G(phi))` costs in bound width.


In [ ]:
# Colab: install the library. Locally, this is a no-op if it is already present.
try:
    import torchmodal
except ImportError:
    !pip install -q torchmodal
    import torchmodal

print('torchmodal', torchmodal.__version__)


In [ ]:
import torch
from torchmodal import MultiAgentKripke, functional as F

torch.manual_seed(0)
kripke = MultiAgentKripke(num_agents=2, num_steps=3, learnable_epistemic=False)
print('states (agent x time):', kripke.num_states)
print('temporal accessibility (forward time flow):')
kripke.A_temporal


## A proposition over spacetime

`isOnline` is true for agent A at every step, and only from step 1 for agent B.


In [ ]:
online = torch.tensor([
    [1.0, 1.0],   # (A, t0)
    [1.0, 1.0],   # (A, t1)
    [1.0, 1.0],   # (A, t2)
    [0.0, 0.0],   # (B, t0)
    [1.0, 1.0],   # (B, t1)
    [1.0, 1.0],   # (B, t2)
])
labels = ['(A,t0)', '(A,t1)', '(A,t2)', '(B,t0)', '(B,t1)', '(B,t2)']


## G and F

`G(phi)` asks whether phi holds at *every* reachable future state; `F(phi)` whether it
holds at *some* reachable future state.


In [ ]:
G = kripke.G(online)
Fin = kripke.F(online)

print(f"{'state':>8}  {'online':>7}  {'G(online)':>18}  {'F(online)':>18}")
for i, lab in enumerate(labels):
    print(f'{lab:>8}  {online[i,0]:>7.0f}  '
          f'[{G[i,0]:.3f}, {G[i,1]:.3f}]      '
          f'[{Fin[i,0]:.3f}, {Fin[i,1]:.3f}]')


`G` collapses at `(B,t0)` — from there the future includes a state where B is offline.
`F` stays high everywhere that can still reach an online state.


## The cost of a composite

`K_G` is `K(G(phi))` — **two** necessity levels. Each level widens the interval by
exactly `tau * H(w)`, so the composite pays twice.


In [ ]:
KG = kripke.K_G(online)
A_full = kripke.get_full_accessibility()

w1 = F.box_width_entropy(kripke.A_temporal, online, tau=0.1)[0]
w2 = F.box_width_entropy(A_full, G, tau=0.1)[0]

print(f'G(online)    at (A,t0): [{G[0,0]:.4f}, {G[0,1]:.4f}]')
print(f'K_G(online)  at (A,t0): [{KG[0,0]:.4f}, {KG[0,1]:.4f}]')
print()
print(f'level 1 width = {w1:.4f}')
print(f'level 2 width = {w2:.4f}')
print(f'total         = {w1 + w2:.4f}   <- two levels, twice the slack')


## Checking the term is still alive

Nesting modal operators costs bound width, and deep enough it costs *everything*: the
lower bound reaches 0 and its gradient vanishes. `gradient_health` finds that.


In [ ]:
from torchmodal.diagnostics import gradient_health

A = torch.ones(8, 8, requires_grad=True)

def nested(depth):
    def run():
        b = torch.ones(8, 2)
        for _ in range(depth):
            b = F.necessity(b, A, tau=0.1)
        return b
    return run

for depth in range(1, 7):
    r = gradient_health(nested(depth), {'A': A})
    lo = r['outputs']['output.L']['min']
    print(f"depth {depth}:  L={lo:.4f}  healthy={str(r['healthy']):5s}  "
          f"{r['issues'][0] if r['issues'] else ''}")


The floor arrives at depth 5, exactly where `k* = ceil(1 / (tau * H))` predicts for a
fully connected 8-world frame. Budget the depth rather than assuming it.
